# FAISS — 현재 권장되는 직접 통합 방식

이 노트북은 책의 FAISS 예제를 **2026-09-21 기준**으로 다시 구성한 것입니다.

2026년 6월 `langchain-community`가 종료·아카이브되었고, 현재 FAISS용 공식 독립 LangChain 패키지는 없습니다. 따라서 수명이 끝난 래퍼를 새 코드에 복사하지 않고, **FAISS 공식 Python API + `langchain-core`의 `BaseRetriever`**를 얇게 연결합니다. 어댑터 코드가 길어 보이지만 저장 형식·점수 의미·필터 동작을 숨기지 않아 학습에는 오히려 유리합니다.

학습 목표:

- cosine 검색용 `IndexFlatIP` 생성과 벡터 정규화
- 유사도 점수, metadata 필터, MMR
- 문서 추가·삭제·병합
- pickle 없는 안전한 로컬 저장/복원
- LangChain `invoke()`/`ainvoke()` Retriever 연결

참고:

- [FAISS 공식 문서](https://faiss.ai/)
- [langchain-community 종료 공지](https://github.com/langchain-ai/langchain-community/issues/674)


## 1. 설치


In [ ]:
%pip install -qU           "langchain-core>=1,<2" "langchain-openai>=1,<2"           "langchain-text-splitters>=1,<2" "faiss-cpu>=1.12,<2"           numpy python-dotenv


## 2. 환경 변수와 버전


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.getenv("LANGSMITH_API_KEY"):
    os.environ.setdefault("LANGSMITH_TRACING", "true")
    os.environ.setdefault("LANGSMITH_PROJECT", "vectorstores-faiss-modern")

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(".env 또는 환경 변수에 OPENAI_API_KEY를 설정하세요.")


In [ ]:
from importlib.metadata import version

for package in ("langchain-core", "langchain-openai", "faiss-cpu"):
    print(f"{package}: {version(package)}")


## 3. 문서 준비

단순 텍스트 파일은 별도 로더 없이 `pathlib`로 읽습니다. 로드와 분할을 분리해 입력 형식이 바뀌어도 분할 정책을 재사용합니다.


In [ ]:
from pathlib import Path

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

data_dir = Path("data")
book_files = [data_dir / "nlp-keywords.txt", data_dir / "finance-keywords.txt"]

if all(path.exists() for path in book_files):
    raw_documents = [
        Document(
            page_content=path.read_text(encoding="utf-8"),
            metadata={"source": path.name, "topic": path.stem.split("-")[0]},
        )
        for path in book_files
    ]
else:
    raw_documents = [
        Document(
            page_content="TF-IDF는 단어 빈도와 역문서 빈도를 곱해 문서에서 중요한 단어를 찾는다.",
            metadata={"source": "nlp-keywords.txt", "topic": "nlp"},
        ),
        Document(
            page_content="Word2Vec은 단어의 문맥을 학습해 의미가 비슷한 단어를 가까운 벡터로 표현한다.",
            metadata={"source": "nlp-keywords.txt", "topic": "nlp"},
        ),
        Document(
            page_content="임베딩 기반 검색은 질문과 문서의 의미적 유사도를 벡터 거리로 계산한다.",
            metadata={"source": "nlp-keywords.txt", "topic": "nlp"},
        ),
        Document(
            page_content="ESG는 환경, 사회, 지배구조 요소를 기업 평가와 투자 판단에 반영한다.",
            metadata={"source": "finance-keywords.txt", "topic": "finance"},
        ),
        Document(
            page_content="시장 금리가 오르면 기존 고정금리 채권의 가격은 일반적으로 하락한다.",
            metadata={"source": "finance-keywords.txt", "topic": "finance"},
        ),
        Document(
            page_content="분산 투자는 상관관계가 다른 자산을 조합해 포트폴리오 위험을 낮춘다.",
            metadata={"source": "finance-keywords.txt", "topic": "finance"},
        ),
    ]

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
documents = splitter.split_documents(raw_documents)
for index, document in enumerate(documents):
    document.metadata["chunk"] = index

nlp_documents = [doc for doc in documents if doc.metadata["topic"] == "nlp"]
finance_documents = [doc for doc in documents if doc.metadata["topic"] == "finance"]
len(nlp_documents), len(finance_documents)


## 4. 작은 FAISS 저장소 어댑터

`IndexFlatIP`에 L2 정규화된 벡터를 넣으면 inner product가 cosine similarity와 같습니다. 따라서 이 노트북의 raw score는 **클수록 유사**하며 범위는 대략 -1~1입니다.

문서 본문과 metadata는 JSON, 벡터 인덱스는 FAISS 바이너리로 저장합니다. 출처 불명의 pickle을 역직렬화할 필요가 없습니다.


In [ ]:
import json
from collections.abc import Callable, Sequence
from typing import Literal
from uuid import uuid4

import faiss
import numpy as np
from langchain_core.embeddings import Embeddings
from langchain_core.retrievers import BaseRetriever
from langchain_core.vectorstores.utils import maximal_marginal_relevance
from pydantic import ConfigDict, Field


MetadataFilter = dict[str, object] | Callable[[dict], bool] | None


class LocalFaissStore:
    '''작고 명시적인 cosine FAISS 저장소 예제.'''

    def __init__(self, embeddings: Embeddings, dimension: int):
        self.embeddings = embeddings
        self.index = faiss.IndexFlatIP(dimension)
        self.documents: list[Document] = []
        self.ids: list[str] = []

    @classmethod
    def from_documents(
        cls,
        documents: Sequence[Document],
        embeddings: Embeddings,
        ids: Sequence[str] | None = None,
    ) -> "LocalFaissStore":
        if not documents:
            raise ValueError("documents는 비어 있을 수 없습니다.")
        vectors = np.asarray(
            embeddings.embed_documents([doc.page_content for doc in documents]),
            dtype="float32",
        )
        store = cls(embeddings=embeddings, dimension=vectors.shape[1])
        store._add_vectors(documents, vectors, ids)
        return store

    @staticmethod
    def _matches(metadata: dict, filter_: MetadataFilter) -> bool:
        if filter_ is None:
            return True
        if callable(filter_):
            return bool(filter_(metadata))
        return all(metadata.get(key) == value for key, value in filter_.items())

    def _add_vectors(
        self,
        documents: Sequence[Document],
        vectors: np.ndarray,
        ids: Sequence[str] | None,
    ) -> list[str]:
        new_ids = list(ids) if ids is not None else [str(uuid4()) for _ in documents]
        if len(documents) != len(new_ids) or len(documents) != len(vectors):
            raise ValueError("documents, ids, vectors 길이가 같아야 합니다.")
        if len(set(new_ids)) != len(new_ids) or set(new_ids) & set(self.ids):
            raise ValueError("문서 ID는 저장소 전체에서 고유해야 합니다.")

        normalized = np.ascontiguousarray(vectors, dtype="float32")
        faiss.normalize_L2(normalized)
        self.index.add(normalized)
        for id_, document in zip(new_ids, documents):
            self.ids.append(id_)
            self.documents.append(
                Document(
                    id=id_,
                    page_content=document.page_content,
                    metadata=dict(document.metadata),
                )
            )
        return new_ids

    def add_documents(
        self,
        documents: Sequence[Document],
        ids: Sequence[str] | None = None,
    ) -> list[str]:
        if not documents:
            return []
        vectors = np.asarray(
            self.embeddings.embed_documents([doc.page_content for doc in documents]),
            dtype="float32",
        )
        return self._add_vectors(documents, vectors, ids)

    def get_by_ids(self, ids: Sequence[str]) -> list[Document]:
        by_id = dict(zip(self.ids, self.documents))
        return [by_id[id_] for id_ in ids if id_ in by_id]

    def _query_vector(self, query: str) -> np.ndarray:
        vector = np.asarray([self.embeddings.embed_query(query)], dtype="float32")
        faiss.normalize_L2(vector)
        return vector

    def _candidate_rows(
        self,
        query_vector: np.ndarray,
        fetch_k: int,
        filter_: MetadataFilter,
    ) -> list[tuple[int, float]]:
        if self.index.ntotal == 0:
            return []
        count = min(max(fetch_k, 1), self.index.ntotal)
        scores, rows = self.index.search(query_vector, count)
        return [
            (int(row), float(score))
            for row, score in zip(rows[0], scores[0])
            if row >= 0 and self._matches(self.documents[int(row)].metadata, filter_)
        ]

    def similarity_search_with_score(
        self,
        query: str,
        k: int = 4,
        *,
        fetch_k: int = 20,
        filter: MetadataFilter = None,
    ) -> list[tuple[Document, float]]:
        candidates = self._candidate_rows(self._query_vector(query), fetch_k, filter)
        return [(self.documents[row], score) for row, score in candidates[:k]]

    def similarity_search(self, query: str, k: int = 4, **kwargs) -> list[Document]:
        return [doc for doc, _ in self.similarity_search_with_score(query, k, **kwargs)]

    def similarity_search_with_relevance_scores(
        self, query: str, k: int = 4, **kwargs
    ) -> list[tuple[Document, float]]:
        return [
            (doc, max(0.0, min(1.0, (score + 1.0) / 2.0)))
            for doc, score in self.similarity_search_with_score(query, k, **kwargs)
        ]

    def max_marginal_relevance_search(
        self,
        query: str,
        k: int = 4,
        *,
        fetch_k: int = 20,
        lambda_mult: float = 0.5,
        filter: MetadataFilter = None,
    ) -> list[Document]:
        query_vector = self._query_vector(query)
        candidates = self._candidate_rows(query_vector, fetch_k, filter)
        if not candidates:
            return []
        rows = [row for row, _ in candidates]
        candidate_vectors = np.vstack([self.index.reconstruct(row) for row in rows])
        selected = maximal_marginal_relevance(
            query_embedding=query_vector[0],
            embedding_list=candidate_vectors,
            k=min(k, len(rows)),
            lambda_mult=lambda_mult,
        )
        return [self.documents[rows[position]] for position in selected]

    def delete(self, ids: Sequence[str]) -> bool:
        targets = set(ids)
        keep_rows = [row for row, id_ in enumerate(self.ids) if id_ not in targets]
        if len(keep_rows) == len(self.ids):
            return False
        vectors = [self.index.reconstruct(row) for row in keep_rows]
        self.ids = [self.ids[row] for row in keep_rows]
        self.documents = [self.documents[row] for row in keep_rows]
        self.index.reset()
        if vectors:
            self.index.add(np.ascontiguousarray(np.vstack(vectors), dtype="float32"))
        return True

    def save_local(self, folder_path: str | Path, index_name: str = "index") -> None:
        folder = Path(folder_path)
        folder.mkdir(parents=True, exist_ok=True)
        faiss.write_index(self.index, str(folder / f"{index_name}.faiss"))
        payload = {
            "ids": self.ids,
            "documents": [
                {"page_content": doc.page_content, "metadata": doc.metadata}
                for doc in self.documents
            ],
        }
        (folder / f"{index_name}.json").write_text(
            json.dumps(payload, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

    @classmethod
    def load_local(
        cls,
        folder_path: str | Path,
        embeddings: Embeddings,
        index_name: str = "index",
    ) -> "LocalFaissStore":
        folder = Path(folder_path)
        index = faiss.read_index(str(folder / f"{index_name}.faiss"))
        payload = json.loads((folder / f"{index_name}.json").read_text(encoding="utf-8"))
        if index.ntotal != len(payload["ids"]) or index.ntotal != len(payload["documents"]):
            raise ValueError("FAISS 인덱스와 JSON 문서 수가 다릅니다.")
        store = cls(embeddings=embeddings, dimension=index.d)
        store.index = index
        store.ids = list(payload["ids"])
        store.documents = [
            Document(id=id_, page_content=item["page_content"], metadata=item["metadata"])
            for id_, item in zip(store.ids, payload["documents"])
        ]
        return store

    def merge_from(self, other: "LocalFaissStore") -> None:
        if self.index.d != other.index.d:
            raise ValueError("두 인덱스의 embedding 차원이 다릅니다.")
        if set(self.ids) & set(other.ids):
            raise ValueError("두 저장소에 중복 문서 ID가 있습니다.")
        if other.index.ntotal:
            vectors = np.vstack(
                [other.index.reconstruct(row) for row in range(other.index.ntotal)]
            )
            self.index.add(np.ascontiguousarray(vectors, dtype="float32"))
        self.ids.extend(other.ids)
        self.documents.extend(other.documents)

    def as_retriever(
        self,
        search_type: Literal["similarity", "mmr", "similarity_score_threshold"] = "similarity",
        search_kwargs: dict | None = None,
    ) -> "LocalFaissRetriever":
        return LocalFaissRetriever(
            store=self,
            search_type=search_type,
            search_kwargs=search_kwargs or {},
        )


class LocalFaissRetriever(BaseRetriever):
    store: LocalFaissStore
    search_type: Literal["similarity", "mmr", "similarity_score_threshold"] = "similarity"
    search_kwargs: dict = Field(default_factory=dict)

    model_config = ConfigDict(arbitrary_types_allowed=True)

    def _get_relevant_documents(self, query: str, *, run_manager, **kwargs) -> list[Document]:
        params = {**self.search_kwargs, **kwargs}
        if self.search_type == "mmr":
            return self.store.max_marginal_relevance_search(query, **params)
        if self.search_type == "similarity_score_threshold":
            threshold = float(params.pop("score_threshold", 0.5))
            return [
                doc
                for doc, relevance in self.store.similarity_search_with_relevance_scores(
                    query, **params
                )
                if relevance >= threshold
            ]
        return self.store.similarity_search(query, **params)


## 5. 저장소 생성

임베딩 모델명을 명시하고 문서 ID를 애플리케이션에서 관리합니다. `from_documents()`는 실제 임베딩 결과로 차원을 알아내므로 숫자를 하드코딩하지 않습니다.


In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
nlp_ids = [f"nlp-{i:03d}" for i in range(len(nlp_documents))]
vector_store = LocalFaissStore.from_documents(nlp_documents, embeddings, nlp_ids)

print("FAISS vector count:", vector_store.index.ntotal)
vector_store.get_by_ids(nlp_ids[:2])


### FAISS 저수준 API 확인

아래 셀은 어댑터가 하는 핵심 작업만 드러냅니다. 벡터를 정규화하지 않으면 inner product를 cosine similarity로 해석할 수 없습니다.


In [ ]:
probe_vectors = np.asarray(
    embeddings.embed_documents(["첫 문장", "두 번째 문장"]),
    dtype="float32",
)
faiss.normalize_L2(probe_vectors)
raw_index = faiss.IndexFlatIP(probe_vectors.shape[1])
raw_index.add(probe_vectors)

raw_query = np.asarray([embeddings.embed_query("첫 번째")], dtype="float32")
faiss.normalize_L2(raw_query)
raw_scores, raw_rows = raw_index.search(raw_query, k=2)
raw_scores, raw_rows


## 6. 유사도 검색, 점수, metadata 필터


In [ ]:
query = "TF-IDF를 설명해 주세요"
for document, cosine in vector_store.similarity_search_with_score(query, k=2):
    print(f"cosine={cosine:.4f}", document.metadata, document.page_content)


In [ ]:
for document, relevance in vector_store.similarity_search_with_relevance_scores(query, k=2):
    print(f"relevance={relevance:.4f}", document.metadata, document.page_content)


FAISS는 metadata를 색인하지 않습니다. 먼저 `fetch_k`개 벡터 후보를 찾은 뒤 애플리케이션에서 필터링하므로, 조건이 선택적일수록 `fetch_k`를 크게 잡아야 합니다. 이 예제 어댑터는 단순 동등 비교 dict 또는 callable을 받습니다.


In [ ]:
all_ids = [f"all-{i:03d}" for i in range(len(documents))]
all_store = LocalFaissStore.from_documents(documents, embeddings, all_ids)

all_store.similarity_search(
    "기업의 지속 가능성",
    k=2,
    fetch_k=10,
    filter={"topic": "finance"},
)


## 7. 문서 추가와 삭제


In [ ]:
added_ids = all_store.add_documents(
    [
        Document(
            page_content="새로 추가한 벡터 저장소 문서입니다.",
            metadata={"source": "manual", "topic": "demo"},
        )
    ],
    ids=["manual-001"],
)
print("added:", added_ids)
all_store.get_by_ids(["manual-001"])


In [ ]:
deleted = all_store.delete(["manual-001"])
print("deleted:", deleted)
all_store.get_by_ids(["manual-001"])


## 8. pickle 없는 로컬 저장과 복원

FAISS 인덱스는 `.faiss`, 문서와 ID는 사람이 검사할 수 있는 `.json`으로 저장합니다. JSON metadata는 문자열·숫자·불리언·목록·dict처럼 JSON으로 표현 가능한 값만 사용하세요.


In [ ]:
faiss_directory = Path("faiss_db")
all_store.save_local(faiss_directory, index_name="faiss_index")
sorted(path.name for path in faiss_directory.iterdir())


In [ ]:
loaded_store = LocalFaissStore.load_local(
    faiss_directory,
    embeddings=embeddings,
    index_name="faiss_index",
)
loaded_store.index.ntotal


## 9. 저장소 병합


In [ ]:
finance_store = LocalFaissStore.from_documents(
    finance_documents,
    embeddings,
    ids=[f"finance-{i:03d}" for i in range(len(finance_documents))],
)

before = loaded_store.index.ntotal
loaded_store.merge_from(finance_store)
print({"before": before, "after": loaded_store.index.ntotal})


## 10. LangChain Retriever

`BaseRetriever`를 구현하면 LCEL 체인에서도 표준 `invoke()`/`ainvoke()`를 사용할 수 있습니다. MMR의 `lambda_mult`는 1에 가까울수록 쿼리 유사도, 0에 가까울수록 결과 다양성을 중시합니다.


In [ ]:
similarity_retriever = loaded_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)
similarity_retriever.invoke("Word2Vec은 무엇인가요?")


In [ ]:
mmr_retriever = loaded_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 6, "lambda_mult": 0.35},
)
mmr_retriever.invoke("Word2Vec과 임베딩")


In [ ]:
threshold_retriever = loaded_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 4, "fetch_k": 10, "score_threshold": 0.5},
)
threshold_retriever.invoke("ESG 투자")


### 비동기 호출 (Jupyter에서 직접 실행)


In [ ]:
async_results = await similarity_retriever.ainvoke("채권과 금리")
async_results


## 정리

- FAISS는 공식 API를 직접 사용하고 LangChain에는 얇은 `BaseRetriever`로 연결
- cosine 검색: L2 정규화 + `IndexFlatIP`
- metadata 필터는 후보 검색 뒤 적용되므로 `fetch_k`를 함께 조정
- 저장: `.faiss` + JSON으로 구성해 pickle 역직렬화 제거
- 체인 연결: `invoke()`/`ainvoke()`
